# 🎾 Capítulo 2 — La Generación Olvidada
### Zverev, Medvedev, Thiem, Tsitsipas: la generación que llegó en el peor momento posible
**Fuente de datos:** [Jeff Sackmann — tennis_atp](https://github.com/JeffSackmann/tennis_atp)  
**Torneos analizados:** Grand Slam y Masters 1000 (2015-2024)

---
## 1. Imports y Carga de Datos

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from itertools import product

# Cargar datos 2015-2024
anos = list(range(2015, 2025))
dfs = []

for ano in anos:
    url = f"https://raw.githubusercontent.com/JeffSackmann/tennis_atp/master/atp_matches_{ano}.csv"
    df_ano = pd.read_csv(url)
    df_ano['season'] = ano
    dfs.append(df_ano)
    print(f"✅ {ano} cargado — {df_ano.shape[0]} partidos")

df = pd.concat(dfs, ignore_index=True)
print(f"\nTotal partidos: {df.shape[0]}")

---
## 2. Definición de Generaciones

In [ ]:
big3 = ['Roger Federer', 'Rafael Nadal', 'Novak Djokovic']

gen_olvidada = ['Alexander Zverev', 'Daniil Medvedev',
                'Dominic Thiem', 'Stefanos Tsitsipas']

nueva_gen = ['Carlos Alcaraz', 'Jannik Sinner', 'Holger Rune']

todos_anos = list(range(2015, 2025))

def clasificar_v2(nombre):
    if nombre in big3:
        return 'Big 3'
    elif nombre in gen_olvidada:
        return 'Gen Olvidada'
    elif nombre in nueva_gen:
        return 'Nueva Gen'
    else:
        return 'Resto'

# Filtrar Slams y Masters 1000, solo finales
df_grandes = df[df['tourney_level'].isin(['G', 'M'])].copy()
finales = df_grandes[df_grandes['round'] == 'F'].copy()
finales['generacion'] = finales['winner_name'].apply(clasificar_v2)

print("Grupos definidos correctamente ✅")

---
## 3. Finales Jugadas por la Generación Olvidada

In [ ]:
# Partidos donde la gen olvidada fue finalista (ganador o perdedor)
finales_go = df_grandes[
    (df_grandes['round'] == 'F') &
    (df_grandes['winner_name'].isin(gen_olvidada) |
     df_grandes['loser_name'].isin(gen_olvidada))
].copy()

def resultado_go(row):
    return 'Victoria' if row['winner_name'] in gen_olvidada else 'Derrota'

def rival_go(row):
    if row['winner_name'] in gen_olvidada:
        return clasificar_v2(row['loser_name'])
    else:
        return clasificar_v2(row['winner_name'])

finales_go['resultado']   = finales_go.apply(resultado_go, axis=1)
finales_go['rival_grupo'] = finales_go.apply(rival_go, axis=1)
finales_go['jugador_go']  = finales_go.apply(
    lambda r: r['winner_name'] if r['winner_name'] in gen_olvidada else r['loser_name'], axis=1
)

resumen = finales_go.groupby(['jugador_go', 'resultado']).size().reset_index(name='count')
derrotas_rival = finales_go[finales_go['resultado'] == 'Derrota'].groupby(
    ['jugador_go', 'rival_grupo']
).size().reset_index(name='derrotas')

print(resumen)

---
## 4. Visualización 1 — Victorias vs Derrotas y Contra Quién Perdieron

In [ ]:
jugadores = ['Alexander Zverev', 'Daniil Medvedev',
             'Dominic Thiem', 'Stefanos Tsitsipas']

victorias = resumen[resumen['resultado'] == 'Victoria'].set_index('jugador_go')['count']
derrotas  = resumen[resumen['resultado'] == 'Derrota'].set_index('jugador_go')['count']

colores_rival = {
    'Big 3':     '#1a78cf',
    'Nueva Gen': '#ff7f0e',
    'Resto':     '#a0a0a0'
}

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7))
x = range(len(jugadores))
ancho = 0.35

# Gráfico izquierdo: victorias y derrotas totales
ax1.bar([p - ancho/2 for p in x],
        [victorias.get(j, 0) for j in jugadores],
        width=ancho, label='Victorias', color='#2ca02c', alpha=0.85)
ax1.bar([p + ancho/2 for p in x],
        [derrotas.get(j, 0) for j in jugadores],
        width=ancho, label='Derrotas', color='#e8462a', alpha=0.85)

ax1.set_title('Finales jugadas por la Generación Olvidada\nSlams y Masters 1000 (2015-2024)',
              fontsize=13, fontweight='bold')
ax1.set_ylabel('Finales', fontsize=11)
ax1.set_xticks(list(x))
ax1.set_xticklabels(['Zverev', 'Medvedev', 'Thiem', 'Tsitsipas'], fontsize=11)
ax1.legend(fontsize=11)
ax1.grid(True, alpha=0.3, axis='y')

# Gráfico derecho: contra quién perdieron (barras apiladas)
grupos_rivales = ['Big 3', 'Nueva Gen', 'Resto']
bottom = [0] * len(jugadores)

for grupo in grupos_rivales:
    valores = []
    for jugador in jugadores:
        dato = derrotas_rival[
            (derrotas_rival['jugador_go'] == jugador) &
            (derrotas_rival['rival_grupo'] == grupo)
        ]['derrotas'].values
        valores.append(dato[0] if len(dato) > 0 else 0)
    ax2.bar(list(x), valores, bottom=bottom,
            label=grupo, color=colores_rival[grupo], alpha=0.85)
    bottom = [b + v for b, v in zip(bottom, valores)]

ax2.set_title('¿Contra quién perdieron sus finales?\nSlams y Masters 1000 (2015-2024)',
              fontsize=13, fontweight='bold')
ax2.set_ylabel('Derrotas en finales', fontsize=11)
ax2.set_xticks(list(x))
ax2.set_xticklabels(['Zverev', 'Medvedev', 'Thiem', 'Tsitsipas'], fontsize=11)
ax2.legend(fontsize=11)
ax2.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('05_gen_olvidada_finales.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 5. Visualización 2 — Ventana de Dominio por Generación

In [ ]:
todas_gen = ['Big 3', 'Gen Olvidada', 'Nueva Gen', 'Resto']

titulos_gen = finales.groupby(
    ['season', 'generacion']
).size().reset_index(name='titulos')

index_completo2 = pd.DataFrame(
    list(product(todos_anos, todas_gen)),
    columns=['season', 'generacion']
)

titulos_gen_completo = index_completo2.merge(
    titulos_gen, on=['season', 'generacion'], how='left'
)
titulos_gen_completo['titulos'] = titulos_gen_completo['titulos'].fillna(0)

colores_gen = {
    'Big 3':        '#1a78cf',
    'Gen Olvidada': '#e8c32a',
    'Nueva Gen':    '#e8462a',
    'Resto':        '#a0a0a0'
}

fig, ax = plt.subplots(figsize=(14, 7))

for gen in todas_gen:
    datos = titulos_gen_completo[titulos_gen_completo['generacion'] == gen]
    ax.fill_between(datos['season'], datos['titulos'],
                    alpha=0.2, color=colores_gen[gen])
    ax.plot(datos['season'], datos['titulos'],
            marker='o', linewidth=2.5, markersize=8,
            label=gen, color=colores_gen[gen])

ax.axvline(x=2022, color='gray', linestyle='--', alpha=0.5)
ax.text(2022.05, 11.5, 'Retiro Federer', fontsize=8, color='gray')
ax.axvline(x=2024, color='gray', linestyle='--', alpha=0.5)
ax.text(2024.05, 11.5, 'Retiro Nadal', fontsize=8, color='gray')

ax.set_title('Ventana de dominio por generación\nSlams y Masters 1000 (2015-2024)',
             fontsize=14, fontweight='bold')
ax.set_xlabel('Temporada', fontsize=11)
ax.set_ylabel('Títulos', fontsize=11)
ax.set_xticks(todos_anos)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('06_ventana_dominio.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 6. Visualización 3 — Evolución Individual de la Generación Olvidada

In [ ]:
colores_go = {
    'Alexander Zverev':   '#e8462a',
    'Daniil Medvedev':    '#9467bd',
    'Dominic Thiem':      '#2ca02c',
    'Stefanos Tsitsipas': '#ff7f0e'
}

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
axes = axes.flatten()

for idx, jugador in enumerate(gen_olvidada):
    ax = axes[idx]

    # Títulos por año
    datos_jugador = finales[
        finales['winner_name'] == jugador
    ].groupby('season').size().reset_index(name='titulos')

    datos_completo = pd.DataFrame({'season': todos_anos})
    datos_completo = datos_completo.merge(datos_jugador, on='season', how='left')
    datos_completo['titulos'] = datos_completo['titulos'].fillna(0)

    # Finales perdidas por año
    derrotas_jugador = finales_go[
        (finales_go['jugador_go'] == jugador) &
        (finales_go['resultado'] == 'Derrota')
    ].groupby('season').size().reset_index(name='derrotas')

    datos_completo = datos_completo.merge(derrotas_jugador, on='season', how='left')
    datos_completo['derrotas'] = datos_completo['derrotas'].fillna(0)

    ax.bar(datos_completo['season'], datos_completo['titulos'],
           color=colores_go[jugador], alpha=0.85, label='Títulos')
    ax.bar(datos_completo['season'], -datos_completo['derrotas'],
           color='#333333', alpha=0.5, label='Finales perdidas')

    ax.axhline(y=0, color='black', linewidth=0.8)
    ax.axvline(x=2022, color='gray', linestyle='--', alpha=0.5)
    ax.axvline(x=2024, color='gray', linestyle='--', alpha=0.5)

    ax.set_title(jugador, fontsize=12, fontweight='bold',
                 color=colores_go[jugador])
    ax.set_xticks(todos_anos)
    ax.set_xticklabels(todos_anos, rotation=45, fontsize=8)
    ax.set_ylabel('Títulos / Derrotas', fontsize=9)
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3, axis='y')

fig.suptitle('La Generación Olvidada — Títulos y finales perdidas\nSlams y Masters 1000 (2015-2024)',
             fontsize=15, fontweight='bold', y=1.02)

plt.tight_layout()
plt.savefig('07_gen_olvidada_individual.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 7. Conclusiones

La Generación Olvidada (Zverev, Medvedev, Thiem, Tsitsipas) vivió la peor paradoja del tenis moderno:

- **Thiem** es el caso más trágico: sus 4 derrotas en finales grandes fueron todas contra el Big 3. Nunca tuvo otra oportunidad y se retiró por lesiones en 2024 sin ver el cambio de era.
- **Tsitsipas** perdió 5 de sus 6 finales contra el Big 3, aplastado sistemáticamente en los momentos decisivos.
- **Medvedev** fue el más activo con 16 finales jugadas, pero bloqueado primero por el Big 3 y luego por la Nueva Gen.
- **Zverev** es el único jugador que perdió finales contra las 3 generaciones distintas, el símbolo perfecto de una generación atrapada entre dos eras.
- Su único peak colectivo fue **2016**, cuando el Big 3 estaba lesionado. Una ventana prestada, no ganada.

> *"Llegaron, pelearon, perdieron siempre contra los mismos y se fueron sin ver el cambio de era."*